# Daemon (030) Demo

Demonstrate the blocking Unix socket daemon basics, including agent metadata and a simple IPC request.


In [ ]:
%load_ext autoreload
%autoreload 2
import json
import os
import socket
import tempfile
from pathlib import Path

from ciphercache.daemon.server import UnixSocketServer
from ciphercache.daemon.state import DaemonConfig, DaemonState
from ciphercache.ipc.framing import decode_single_frame, encode_message


In [ ]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-daemon-demo-"))
config = DaemonConfig(data_dir=data_dir, write_agent_metadata=True)
state = DaemonState(config=config)
server = UnixSocketServer(config=config, state=state)
server.setup()
agent_path = data_dir / "agent.json"
json.loads(agent_path.read_text(encoding="utf-8"))


In [ ]:
client, server_sock = socket.socketpair()
request = {"version": "v0", "id": "ping", "type": "request", "op": "ping", "payload": {}}
client.sendall(encode_message(request))
server._handle_connection(server_sock)
response = decode_single_frame(client.recv(4096))
client.close()
server_sock.close()
response


In [ ]:
server.close()
os.path.exists(config.socket_path)


In [ ]:
server._handle_signal(15, None)
server.listener.fileno()
